In [ ]:
# Imports and constants
import os, glob, string
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize
LABELS = ["A", "B", "C", "D", "E"]

In [ ]:
# Load training data
def find_data():
    for d in ["/kaggle/input/smart-mcq-solver-challenge", "/kaggle/input/competitions/smart-mcq-solver-challenge",
              "/mnt/user-data/uploads", "."]:
        if os.path.exists(os.path.join(d, "train.csv")): return d
    h = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if h: return os.path.dirname(h[0])
    raise FileNotFoundError
train = pd.read_csv(os.path.join(find_data(), "train.csv"))
print(train.shape, {l: int(train[l].isna().sum()) for l in LABELS})

In [ ]:
# Q1: Answer frequency distribution
counts = {l: int((train["answer"].str.upper() == l).sum()) for l in LABELS}
mx, mn = max(counts.values()), min(counts.values())
print("Q1:", counts, "| most + least =", mx + mn)

In [ ]:
# Q2: Vocabulary size of cleaned prompts
_tbl = str.maketrans("", "", string.punctuation)
def clean(s): return str(s).lower().translate(_tbl).split()
vocab = set()
for p in train["prompt"]: vocab.update(clean(p))
print("Q2:", len(vocab))

In [ ]:
# Q3: Stop-word removal for row ID 1
toks1 = clean(train.loc[train["id"] == 1, "prompt"].iloc[0])
kept = [w for w in toks1 if w not in ENGLISH_STOP_WORDS]
print("Q3:", len(toks1), "->", len(kept))

In [ ]:
# Q4: TF-IDF vocabulary size under two corpus interpretations
docs_A = list(train["prompt"].astype(str))
for l in LABELS: docs_A += list(train[l].astype(str))
docs_B = [" ".join([str(r["prompt"])] + [str(r[l]) for l in LABELS]) for _, r in train.iterrows()]
vecA = TfidfVectorizer(stop_words="english").fit(docs_A)
vecB = TfidfVectorizer(stop_words="english").fit(docs_B)
print("Q4:", len(vecA.vocabulary_), "(A)", len(vecB.vocabulary_), "(B)")

In [ ]:
# Q5: Cosine similarity between prompt and option A for row ID 1
r1 = train.loc[train["id"] == 1].iloc[0]
def cos1(vec, a, b):
    va, vb = normalize(vec.transform([str(a)])), normalize(vec.transform([str(b)]))
    return round(float((va @ vb.T).toarray()[0, 0]), 4)
print("Q5:", cos1(vecA, r1["prompt"], r1["A"]), "(A)", cos1(vecB, r1["prompt"], r1["A"]), "(B)")

In [ ]:
# Per-row prompt-vs-option cosine similarity matrix
def option_sims(vec):
    P = normalize(vec.transform(train["prompt"].astype(str)))
    cols = []
    for l in LABELS:
        O = normalize(vec.transform(train[l].astype(str)))
        cols.append(np.asarray(P.multiply(O).sum(axis=1)).ravel())
    return np.vstack(cols).T
simsA, simsB = option_sims(vecA), option_sims(vecB)
ans_idx = train["answer"].str.upper().map({l: i for i, l in enumerate(LABELS)}).to_numpy()

In [ ]:
# Q6: Top-cosine accuracy
def top1_acc(sims):
    pred = sims.argmax(axis=1)
    return round((pred == ans_idx).mean() * 100, 4)
print("Q6:", top1_acc(simsA), "% (A)", top1_acc(simsB), "% (B)")

In [ ]:
# Q7 and Q8: MAP@3 for specific predictions
def map3(truth, pred): return (1.0 / (pred.index(truth) + 1)) if truth in pred[:3] else 0.0
print("Q7:", map3("C", ["C", "A", "B"]))
print("Q8:", map3("B", ["D", "B", "E"]))

In [ ]:
# Q9: Majority-class baseline MAP@3
order = [k for k, _ in sorted(counts.items(), key=lambda x: -x[1])][:3]
m9 = np.mean([map3(str(a).upper(), order) for a in train["answer"]])
print("Q9:", round(m9, 4))

In [ ]:
# Q10: TF-IDF pipeline MAP@3
def pipeline_map3(sims):
    tot = 0.0
    for i in range(len(train)):
        ranked = list(np.argsort(-sims[i], kind="stable"))
        a = ans_idx[i]
        tot += (1.0 / (ranked.index(a) + 1)) if a in ranked[:3] else 0.0
    return round(tot / len(train), 4)
print("Q10:", pipeline_map3(simsA), "(A)", pipeline_map3(simsB), "(B)")